# Golden Balloon — Android ARM64
Compila el proyecto adjunto en Google Colab (entorno CPU). No necesitas subir una ROM: se importa en el teléfono después.

El botón de ejecución de la celda de compilación instala el SDK/NDK de Android y acepta sus licencias mediante `--install-sdk`. Se recomienda disponer de 8 GB de RAM y 10 GB libres. La primera ejecución necesita Internet.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()  # Selecciona GoldenBalloon-Android-ARM64.zip
archive = next(name for name in uploaded if name.lower().endswith('.zip'))
destination = Path('/content/goldenballoon-arm64')
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        p = Path(item.filename)
        if p.is_absolute() or '..' in p.parts:
            raise ValueError('Ruta no segura en el ZIP')
    z.extractall(destination)
projects = list(destination.rglob('tools/android/build.py'))
assert len(projects) == 1, 'Debe haber exactamente un proyecto en el ZIP'
project = projects[0].parents[2]
print('Proyecto:', project)
del uploaded


In [ ]:
import os, subprocess
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jdk-headless'], check=True)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
# Compilación completa sin Gradle/Maven: usa directamente SDK, NDK, CMake y D8.
log_path = project / 'colab-build.log'
command = ['python3', str(project/'tools/android/build.py'), '--sdk-only', '--install-sdk', '--jobs', '2']
with log_path.open('w') as log:
    process = subprocess.Popen(command, cwd=project, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end=''); log.write(line)
    result = process.wait()
assert result == 0, f'Falló la compilación. Descarga {log_path.name} para revisar el error.'


In [ ]:
apk = project / 'out/GoldenBalloon-arm64-debug.apk'
assert apk.is_file(), 'No se generó el APK; revisa la celda anterior'
files.download(str(apk))


## Registros y símbolos (opcional)
Los símbolos permiten investigar un cierre nativo sin recompilar. Para actualizaciones instalables, conserva también la clave local de depuración; no la publiques. Una nueva sesión de Colab genera otra clave y Android puede rechazar una actualización sobre una APK firmada con la clave anterior. Exporta tus partidas antes de desinstalar.


In [ ]:
# Descomenta el archivo que quieras descargar:
# files.download(str(log_path))
# files.download(str(project/'out/GoldenBalloon-arm64-symbols.zip'))
# files.download(str(project/'.build-tools/goldenballoon-debug.keystore'))
